## Load the File

In [1]:
from pathlib import Path
from bs4 import BeautifulSoup

# Finds the file by pattern instead of hardcoding the date
path = next(Path("../data/raw").glob("AAPL_10-K_*.html"))
html = path.read_text(encoding="utf-8")
# Turns the HTML string into a navigable tree of tags
soup = BeautifulSoup(html, "lxml")

print(path.name, f"{len(html):,} characters")

AAPL_10-K_2026-11-03.html 1,650,108 characters


/var/folders/g0/83td4ygx4q3704543wrw62180000gn/T/ipykernel_24165/2262744922.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


## Remove Hidden XBRL Data

In [2]:
# Finds every tag styled display:none
hidden = soup.find_all(style=lambda s: s and "display:none" in s.replace(" ", ""))
print(f"Hidden elements: {len(hidden)}")

for tag in hidden:
	tag.decompose()   # Removes the tag from the document entirely

Hidden elements: 77


## Extract Text

In [3]:
# Strips all tags nd keeps only the text
text = soup.get_text(separator="\n")
# Trims each line and drops empty ones
lines = [line.strip() for line in text.splitlines() if line.strip()]

print(f"{len(lines):,} lines")
print("\n".join(lines[:40]))

4,162 lines
aapl-20260926
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM
10-K
(Mark One)
☒
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended
September 26
, 2026
or
☐
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from
to
.
Commission File Number:
001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California
94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
MS 927-4GC
Cupertino
,
California
95014
(Address of principal executive offices)
(Zip Code)
(
408
)
996-1010
(Registrant’s telephone number, including area code)


## How Do Tables Look as Plain Text?

In [4]:
# Collects every line number containing "Total net sales"
hits = [i for i, line in enumerate(lines) if "Total net sales" in line]
print(f"{len(hits)} matches")

i = hits[0]
for line in lines[i - 5 : i + 10]:
	print(repr(line))

5 matches
'%'
'33,696'
'10'
'%'
'30,658'
'Total net sales'
'$'
'478,834'
'15'
'%'
'$'
'416,161'
'6'
'%'
'$'


## Extract a Table as Rows

In [5]:
tables = soup.find_all("table")
print(f"{len(tables)} tables")

# Picks the first table containing "Total net sales"
target = next(t for t in tables if "Total net sales" in t.get_text())
for tr in target.find_all("tr"):   # Iterates over the table's rows
	cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
	print(cells)

68 tables
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', '2026', '', 'Change', '', '2025', '', 'Change', '', '2024']
['Americas', '$', '198,083', '', '', '11', '%', '', '$', '178,353', '', '', '7', '%', '', '$', '167,045', '']
['Europe', '127,604', '', '', '15', '%', '', '111,032', '', '', '10', '%', '', '101,328', '']
['Greater China', '82,378', '', '', '28', '%', '', '64,377', '', '', '(4)', '%', '', '66,952', '']
['Japan', '30,908', '', '', '8', '%', '', '28,703', '', '', '15', '%', '', '25,052', '']
['Rest of Asia Pacific', '39,861', '', '', '18', '%', '', '33,696', '', '', '10', '%', '', '30,658', '']
['Total net sales', '$', '478,834', '', '', '15', '%', '', '$', '416,161', '', '', '6', '%', '', '$', '391,035', '']


## Clean a Row

In [11]:
import re

def clean_row(cells):
	out = []
	for c in cells:
		# Removes whitespace right after (. \s+ means one or more whitespace characters.
		# re is Python's regular expression library.
		c = re.sub(r"\(\s+", "(", c.strip())
		# Remove whitespace right before ), turning ( 321 ) into (321)
		c = re.sub(r"\s+\)", ")", c)

		# Skips empty cells and lone $ signs, which carry little meaning and only
		# misalign columns
		if c in ("", "$"):
			continue
		# Attaches lone % or ) to the previous cell
		if c in ("%", ")", ")%") and out:
			out[-1] += c
			continue
		out.append(c)
	return out

## Convert to Markdown

In [14]:
def is_section(row):
	return len(row) == 1 and row[0].endswith(":")

def table_to_markdown(table):
	rows = []
	for tr in table.find_all("tr"):
		cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
		row = clean_row(cells)
		if row:   # Drops rows that are empty after cleaning, like the spacer row
			rows.append(row)

	width = max(len(r) for r in rows)
	n_cols = width - 1

	# 1. Header: leading rows without a label cell (shorter than data rows)
	header = [""] * n_cols
	# Takes rows from the top as headers while they're shorter than data rows, stopping at the first section title
	while rows and len(rows[0]) < width and not is_section(rows[0]):
		hr = rows.pop(0)
		# A single-cell header like "Years Ended" spans all columns, so it's copied to each one
		if len(hr) == 1:
			hr = hr * n_cols
		# Pads short header rows on the left ot right-align them, as before
		hr = [""] * (n_cols - len(hr)) + hr
		# Joins header rows column by column
		header = [f"{h} {x}".strip() for h, x in zip(header, hr)]

	# 2. Body: prefix each row with its section title
	body = []
	section = ""
	for r in rows:
		if is_section(r):
			# Remembers the section title and skips the row itself, since its meaning
			# moves into the labels below
			section = r[0].rstrip(":")
			continue
		label = r[0]
		if section and not label.startswith("Total"):
			label = f"{section} - {label}"
		body.append([label] + r[1:] + [""] * (width - len(r)))
		# A "Total" row closes the section, so the next row doesn't inherit it
		if r[0].startswith("Total"):
			section = ""

	# 3. Markdown
	lines = ["| | " + " | ".join(header) + " |", "|" + " --- |" * width]
	lines += ["| " + " | ".join(r) + " |" for r in body]
	return "\n".join(lines)

print(table_to_markdown(target))

| | 2026 | Change | 2025 | Change | 2024 |
| --- | --- | --- | --- | --- | --- |
| Americas | 198,083 | 11% | 178,353 | 7% | 167,045 |
| Europe | 127,604 | 15% | 111,032 | 10% | 101,328 |
| Greater China | 82,378 | 28% | 64,377 | (4)% | 66,952 |
| Japan | 30,908 | 8% | 28,703 | 15% | 25,052 |
| Rest of Asia Pacific | 39,861 | 18% | 33,696 | 10% | 30,658 |
| Total net sales | 478,834 | 15% | 416,161 | 6% | 391,035 |


In [15]:
candidates = [t for t in tables if "Total net sales" in t.get_text()]
print(f"{len(candidates)} candidate tables\n")

for i, t in enumerate(candidates):
	print(f"===== Table {i} =====")
	print(table_to_markdown(t))
	print()

5 candidate tables

===== Table 0 =====
| | 2026 | Change | 2025 | Change | 2024 |
| --- | --- | --- | --- | --- | --- |
| Americas | 198,083 | 11% | 178,353 | 7% | 167,045 |
| Europe | 127,604 | 15% | 111,032 | 10% | 101,328 |
| Greater China | 82,378 | 28% | 64,377 | (4)% | 66,952 |
| Japan | 30,908 | 8% | 28,703 | 15% | 25,052 |
| Rest of Asia Pacific | 39,861 | 18% | 33,696 | 10% | 30,658 |
| Total net sales | 478,834 | 15% | 416,161 | 6% | 391,035 |

===== Table 1 =====
| | 2026 | Change | 2025 | Change | 2024 |
| --- | --- | --- | --- | --- | --- |
| iPhone | 254,098 | 21% | 209,586 | 4% | 201,183 |
| Mac | 37,929 | 13% | 33,708 | 12% | 29,984 |
| iPad | 27,929 | —% | 28,023 | 5% | 26,694 |
| Wearables, Home and Accessories | 35,577 | —% | 35,686 | (4)% | 37,005 |
| Services (1) | 123,301 | 13% | 109,158 | 14% | 96,169 |
| Total net sales | 478,834 | 15% | 416,161 | 6% | 391,035 |

===== Table 2 =====
| | Years Ended September 26, 2026 | Years Ended September 27, 2025 | Years End